# EOG: dynamic historical research

No financial statement values or fixed year lists are embedded in notebook code.

`FINANCIAL_SOURCE = "wrds"` reads your latest saved WRDS report. `"sec"` reads SEC standard XBRL facts, with an optional online refresh. The chosen source stays explicit because its definitions differ from the other source. The current setting uses the saved SEC download and compares it with your WRDS cache. Cached runs do not prompt for credentials.

The operating data are previously reviewed observations stored separately by ticker, year, metric, units, and source. They are **not automatically refreshed** in that earlier table. Section 9 independently retrieves the six production/price inputs and revenue components from supported annual filing layouts. Unavailable years stay blank, so old values can never be relabeled as a new fiscal year.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "app/history.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the DCF-Model project folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reload edited project modules before rebuilding the annual DataFrames.
# Jupyter otherwise keeps previously imported Python code in memory.
import importlib
import app.sec as sec_module
import app.history as history_module
import app.history_review as review_module

if Path(history_module.__file__).resolve() != PROJECT_ROOT / "app/history.py":
    raise RuntimeError("A different app.history is loaded. Restart the kernel from this project and Run All.")

importlib.reload(sec_module)
importlib.reload(history_module)
importlib.reload(review_module)

from app.history import load_history, load_operating_observations, HistoryError
from app.sec import SECError

TICKER = "EOG"
HISTORY_YEARS = 5
END_YEAR = None                # None = newest annual period in the selected source
FINANCIAL_SOURCE = "sec"       # Audited filing data; WRDS stays available for comparison
SEC_CIK = "0000821189"         # Company identifier; change with ticker for SEC
REFRESH_SEC = False           # Latest SEC download is cached; True requests a new snapshot

history = load_history(
    ticker=TICKER, years=HISTORY_YEARS, end_year=END_YEAR,
    provider=FINANCIAL_SOURCE, cik=SEC_CIK,
    refresh=REFRESH_SEC if FINANCIAL_SOURCE == "sec" else False,
    root=PROJECT_ROOT,
)
annual = history.annual
print(f"{TICKER}: {annual.index.min()} through {annual.index.max()}")
print(f"Provider: {history.provider} | retrieved UTC: {history.retrieved_at}")
print(f"Raw source: {history.raw_path.relative_to(PROJECT_ROOT)}")
for message in history.warnings:
    print("NOTE:", message)


## 1. Standardized financial history

Amounts are USD millions, except diluted weighted-average shares in millions. Names describe a common schema; the audit table records the actual provider field. EBITDA is operating income plus the selected provider's D&A. Compustat revenue, D&A, debt, and cash can have different definitions from the filed statement lines.

The expanded WRDS refresh requests balance-sheet and tax inputs. The coverage and reconciliation sections below show whether they are present in the selected report. Missing fields remain blank.
For EOG, SEC capex is oil-and-gas property **and equipment** additions plus other PP&E additions. Mineral-interest purchases alone are not the total. Where SEC does not tag total liabilities, the EOG adapter sums current liabilities, noncurrent debt, other noncurrent liabilities, and deferred tax liabilities. The audit table shows these formulas and their component sources.


In [ ]:
financial_fields = [
    "revenue", "operating_expenses", "operating_expenses_before_da",
    "ebit", "dd_a", "ebitda", "interest_expense", "pretax_income",
    "income_tax", "net_income", "operating_cash_flow", "capex",
    "cash", "receivables", "inventory", "accounts_payable",
    "debt_current", "debt_noncurrent", "total_debt", "diluted_shares",
]
history_table = annual[financial_fields].T
history_table.index.name = "Standard field"
display(history_table.round(2))

## 2. Input coverage and source audit

This shows gaps before calculations. Each retrieved value has an original field/tag, reporting period, raw value, unit scale, retrieval time, and filing reference where available. Calculated entries identify their component fields. SEC selection uses actual period dates and the latest available annual disclosure; this is not a point-in-time backtest. Company custom XBRL tags need a separate mapping and are left unavailable.

In [ ]:
coverage = pd.DataFrame({
    "Available years": annual[financial_fields].notna().sum(),
    "Requested years": len(annual),
})
coverage["Missing years"] = [
    ", ".join(map(str, annual.index[annual[field].isna()]))
    for field in financial_fields
]
display(coverage)

audit_columns = [
    "year", "metric", "value", "units", "provider", "source_field",
    "original_value", "original_units", "period_start", "period_end",
    "filed", "retrieved_at_utc", "source", "status",
]
display(history.audit.reindex(columns=audit_columns))

## 2a. Accounting reconciliation

These checks compare related fields in the selected source for every displayed year. Residuals are in USD millions. A missing input cannot pass; a difference stays visible for review and is never automatically adjusted. The default tolerance is $0.05 million.

A tie confirms internal consistency. It does not establish that Compustat definitions match EOG's filing lines. Discontinued operations or noncontrolling interests may explain differences in these simplified bridges for other companies or periods. Debt scope and operating working capital still require source review.


In [ ]:
from app.history_review import accounting_checks

reconciliation_checks = accounting_checks(history)
display(reconciliation_checks)
print(reconciliation_checks["Status"].value_counts().to_string())

key_inputs = [
    "receivables", "inventory", "accounts_payable", "pretax_income",
    "income_tax", "debt_current", "debt_noncurrent", "change_nwc_proxy",
]
input_review = pd.DataFrame({
    "Years available": annual[key_inputs].notna().sum(),
    "Years requested": len(annual),
})
display(input_review)
print("NWC remains a balance-sheet proxy: total receivables + inventory - accounts payable.")
print("Review non-operating balances, acquisition effects and capital payables before UFCF.")


## 3. Financial ratios, debt and working-capital diagnostics

`total_debt` is the sum of the selected provider's current and noncurrent debt. It is not automatically a valuation debt balance. In the saved EOG extract, Compustat includes operating leases in these fields; do not add them again. SEC debt and lease tags are displayed separately for reconciliation.

The working-capital proxy is **receivables + inventory - accounts payable**. A positive annual change is an increased balance-sheet investment. Acquisition balances, joint-interest accounts, capital payables, and omitted current accounts still require review before treating this as DCF operating NWC. The loader retains one preceding year for changes; missing prior-year inputs stay blank.

In [ ]:
ratio_fields = [
    "revenue_growth_pct", "ebit_margin_pct", "ebitda_margin_pct",
    "effective_tax_rate_pct", "capex_revenue_pct",
]
display(annual[ratio_fields].T.round(2))

debt_fields = [
    "debt_current", "debt_noncurrent", "total_debt", "cash", "net_debt",
    "operating_leases_current", "operating_leases_noncurrent",
    "finance_leases_current", "finance_leases_noncurrent",
]
display(annual[debt_fields].T.round(2))

working_capital = annual[[
    "receivables", "inventory", "accounts_payable",
    "nwc_proxy", "change_nwc_proxy",
]]
display(working_capital.round(2))

## 4. Operating history aligned to the selected years

These observations were migrated from the prior notebook into a source dataset. This removes duplicated arrays and fixes their year alignment; it does not automate retrieval of new production, price, reserve, or WTI observations. A new fiscal year shows missing operating data until a reviewed source record is added. Source and definition notes remain visible below.

In [ ]:
operating, operating_audit = load_operating_observations(
    PROJECT_ROOT / "data/reference/eog_operating_history.json",
    years=annual.index, ticker=TICKER,
)
required_operating = [
    "oil_mbbl_day", "gas_mmcf_day", "ngl_mbbl_day", "total_mboe_day",
    "annual_mmboe", "realized_oil", "realized_gas", "realized_ngl",
    "loe_boe", "gpt_boe", "ga_boe", "taxes_other_than_income",
    "oil_reserves_mmbbl", "ngl_reserves_mmbbl", "gas_reserves_bcf",
    "reserves_mmboe", "wti",
]
operating = operating.reindex(columns=required_operating)
display(operating.T.round(2))
display(operating_audit)

missing_operating = operating.isna().sum(axis=1)
for year, missing in missing_operating.items():
    if missing:
        print(f"FY{year}: {missing} operating fields unavailable; new disclosures needed.")

## 5. Business-driver metrics

All calculations join on fiscal year, with no positional list matching or forward-filling. Production growth for the first displayed year is blank here because this operating table only contains displayed years. Gas is converted at EOG's disclosed 6 Mcf per Boe. EBITDA per Boe uses the selected financial provider and includes that provider's definition limitations.

In [ ]:
drivers = pd.DataFrame(index=annual.index)
total = operating["total_mboe_day"].replace(0, float("nan"))
volume = operating["annual_mmboe"].replace(0, float("nan"))
drivers["Oil mix (%)"] = 100 * operating["oil_mbbl_day"] / total
drivers["Gas mix (%)"] = 100 * (operating["gas_mmcf_day"] / 6) / total
drivers["NGL mix (%)"] = 100 * operating["ngl_mbbl_day"] / total
drivers["Annual production growth (%)"] = 100 * (volume / volume.shift() - 1)
drivers["LOE / Boe (USD)"] = operating["loe_boe"]
drivers["GP&T / Boe (USD)"] = operating["gpt_boe"]
drivers["G&A / Boe (USD)"] = operating["ga_boe"]
drivers["Capex / Boe (USD)"] = annual["capex"] / volume
drivers["Calculated EBITDA / Boe (USD)"] = annual["ebitda"] / volume
drivers["WTI less realized oil (USD/Bbl)"] = operating["wti"] - operating["realized_oil"]
display(drivers.T.round(2))

## 6. Compare the two sources when both caches exist

The comparison uses the same selected years. A missing SEC cache does not prevent exploration of WRDS data. Differences remain labeled for investigation; the notebook does not change the chosen source to make differences disappear. The debt bridge tests whether operating leases explain the observed gap.

In [ ]:
comparison_provider = "sec" if FINANCIAL_SOURCE == "wrds" else "wrds"
comparison_history = None
try:
    comparison_history = load_history(
        TICKER, HISTORY_YEARS, provider=comparison_provider, cik=SEC_CIK,
        end_year=int(annual.index.max()), refresh=False, root=PROJECT_ROOT,
    )
except (HistoryError, SECError) as error:
    print(f"Comparison unavailable: {error}")

if comparison_history is not None:
    sec_annual = annual if FINANCIAL_SOURCE == "sec" else comparison_history.annual
    wrds_annual = annual if FINANCIAL_SOURCE == "wrds" else comparison_history.annual
    comparison_rows = []
    for metric in ["revenue", "ebit", "dd_a", "capex", "operating_cash_flow"]:
        for year in annual.index:
            vendor = wrds_annual.loc[year, metric]
            filed = sec_annual.loc[year, metric]
            comparison_rows.append({
                "Year": year, "Metric": metric, "WRDS": vendor, "SEC": filed,
                "WRDS less SEC": vendor - filed,
            })
    display(pd.DataFrame(comparison_rows).round(2))
    debt_bridge = pd.DataFrame({
        "WRDS debt": wrds_annual["total_debt"],
        "SEC debt": sec_annual["total_debt"],
        "SEC operating leases": sec_annual["operating_leases"],
    })
    debt_bridge["Unexplained difference"] = (
        debt_bridge["WRDS debt"] - debt_bridge["SEC debt"]
        - debt_bridge["SEC operating leases"]
    )
    display(debt_bridge.round(2))

## 6a. Receivables classification and acquisition review

The SEC download uses EOG's reported receivables separately from income taxes receivable. Compustat reports total receivables. A difference matching the tax receivable is evidence of a classification difference; it does not justify applying a blanket adjustment to every year.

The debt bridge above compares Compustat debt with SEC debt plus operating leases. Finance leases are already included in EOG's reported debt. Keep that classification consistent when selecting a valuation debt balance.

Cash paid for business acquisitions is displayed separately from capital expenditures. Year-end NWC movements in acquisition years can include balances acquired with the business. The NWC proxy therefore remains under review before UFCF; no acquisition-related working-capital adjustment is guessed here.


In [ ]:
if comparison_history is not None:
    if "income_taxes_receivable" not in sec_annual.columns:
        raise RuntimeError(
            "The SEC table predates the tax-receivable field. "
            "Run the setup cell and every cell above this one again, "
            "or choose Restart Kernel and Run All."
        )
    receivables_review = pd.DataFrame({
        "WRDS total receivables": wrds_annual["receivables"],
        "SEC reported receivables": sec_annual["receivables"],
        "SEC income taxes receivable": sec_annual["income_taxes_receivable"],
    })
    receivables_review["WRDS less SEC receivables"] = (
        receivables_review["WRDS total receivables"]
        - receivables_review["SEC reported receivables"]
    )
    receivables_review["Difference less reported tax receivable"] = (
        receivables_review["WRDS less SEC receivables"]
        - receivables_review["SEC income taxes receivable"]
    )
    display(receivables_review.round(2))
    print("A zero residual supports the tax-receivable explanation for that year; missing tax data stay blank.")

display(annual[["capex", "acquisitions", "nwc_proxy", "change_nwc_proxy"]].round(2))
acquisition_years = annual.index[annual["acquisitions"].notna() & annual["acquisitions"].ne(0)].tolist()
print("Years with reported acquisition cash flows requiring NWC review:", acquisition_years)


## 7. Historical operating review

Compare the latest selected year with the immediately preceding year. Every value comes from the normalized financial history or the year-indexed operating observations. Absolute changes retain the row's units; changes in margins and production mix are shown in percentage points.

The selected unit-cost subtotal is lease and well + gathering, processing and transportation + G&A. It excludes DD&A, exploration, dry-hole costs, impairments, marketing costs, and taxes other than income. It is not total operating cost. Historical growth includes acquired operations where applicable and is not automatically a forecast assumption.

Use this review to identify which inputs require a separate forecast decision: organic production by commodity, realized prices and differentials, recurring unit costs, capital requirements, and operating working capital. Operating observation retrieval is still a separate step; a missing new year remains blank.


In [ ]:
latest_year = int(annual.index.max())
previous_year = latest_year - 1

if previous_year in annual.index:
    for metric, label in [
        ("revenue", "Revenue"), ("operating_cash_flow", "Operating cash flow"),
        ("capex", "Capital expenditures"),
    ]:
        latest, previous = annual.loc[latest_year, metric], annual.loc[previous_year, metric]
        if pd.notna(latest) and pd.notna(previous) and previous != 0:
            print(f"{label}: {100 * (latest / previous - 1):+.1f}% in FY{latest_year} versus FY{previous_year}.")
        else:
            print(f"{label}: insufficient data for a year-over-year comparison.")
    print(f"Provider used: {history.provider}; inspect reconciliation before mixing sources.")
else:
    print("Select at least two years to compare trends.")


from app.history_review import operating_driver_review

operating_review = operating_driver_review(annual, operating)
display(operating_review.round(2))

for metric in ["Annual production", "Realized oil price", "Realized gas price",
               "Selected unit-cost subtotal", "Capex per Boe", "Calculated EBITDA per Boe"]:
    change = operating_review.loc[metric, "Change (%)"]
    if pd.notna(change):
        print(f"{metric}: {change:+.1f}% versus the preceding year.")
    else:
        print(f"{metric}: comparison unavailable; inspect source coverage.")

margin_change = operating_review.loc["Operating margin", "Change"]
if pd.notna(margin_change):
    print(f"Operating margin changed {margin_change:+.2f} percentage points.")

acquisition_cash = annual.loc[latest_year, "acquisitions"]
if pd.notna(acquisition_cash) and acquisition_cash != 0:
    print(f"FY{latest_year} includes reported acquisition cash flows of ${acquisition_cash:,.0f} million.")
    print("Separate acquired production and acquired working capital before treating the changes as recurring.")
elif pd.isna(acquisition_cash):
    print("Acquisition cash-flow data are missing; organic growth has not been established.")
else:
    print("No acquisition cash outflow is reported in this row; that alone does not establish organic growth.")

print("Capex per Boe uses total annual production. A lower ratio alone does not establish lower development costs.")
print("Next: quantify acquisition effects and finalize operating NWC before setting forecast assumptions.")


## 8. Acquisition effects: establish the historical base for forecasting

An acquisition changes production, expenses and balance-sheet accounts on different timelines. We first explain those effects before choosing recurring FCF assumptions.

This section reads **the latest selected year's annual filing**, resolved from SEC submissions metadata. Its inline XBRL includes acquisition dimensions that the companyfacts API omits. The company adapter maps tags and transaction contexts; it does not contain financial amounts, acquisition dates or fixed year ranges.

The document is cached separately from companyfacts. `REFRESH_FILING = False` uses the saved document. For a newly selected filing, set it to `True` once, then back to `False`. Refresh SEC metadata in the setup cell first to discover newly filed reports. Unsupported or ambiguous disclosures need review; a missing deal is not evidence that no acquisition occurred.


In [ ]:
import app.sec_filings as filings_module
import app.acquisition_review as acquisition_module
importlib.reload(filings_module)
importlib.reload(acquisition_module)

REFRESH_FILING = False
acquisition_tables = {}
acquisition_audit = pd.DataFrame()

try:
    if TICKER.upper() != "EOG" or history.provider.upper() != "SEC":
        raise ValueError("This filing adapter currently requires EOG and FINANCIAL_SOURCE = 'sec'.")
    filing_bytes, filing_metadata, filing_path = filings_module.load_annual_filing(
        SEC_CIK, TICKER, int(annual.index.max()),
        refresh=REFRESH_FILING, root=PROJECT_ROOT,
    )
    acquisition_deals, acquisition_facts, acquisition_audit = acquisition_module.read_eog_acquisitions(
        filing_bytes, filing_metadata,
    )
    acquisition_tables = acquisition_module.acquisition_bridges(
        annual, operating, acquisition_deals, acquisition_facts, provider=history.provider,
    )
    print(f"Fiscal period end: {filing_metadata['report_date']} | Filed: {filing_metadata['filed']}")
    print(f"Retrieved UTC: {filing_metadata['retrieved_at_utc']}")
    display(Markdown(f"Source: [EOG annual filing]({filing_metadata['url']})"))
    print(f"Saved document: {filing_path.relative_to(PROJECT_ROOT)}")
    if acquisition_tables:
        display(acquisition_tables["deals"].round(2))
    else:
        print("No supported current-year acquisition was mapped. Review the filing before concluding there were none.")
except (SECError, ValueError, OSError) as error:
    print(f"Acquisition review unavailable: {error}")


### 8a. Acquisition cash and working-capital bridge

The bridge first compares mapped consideration **less acquired cash** with reported acquisition cash outflows. If they do not reconcile, the residual NWC calculation stays unavailable.

For the same narrow definition used above:

- Acquired NWC proxy = acquired receivables + inventory - acquired payables.
- Remaining NWC change = reported annual change - acquired NWC proxy.

A negative remaining change indicates a release in this balance-sheet proxy. It is **not yet the DCF cash-flow input**: capital payables, noncash movements, joint-interest balances and omitted operating accounts remain to be reviewed. A cash tie supports coverage of mapped cash acquisitions; it does not establish that all noncash transactions or other balance changes have been captured.

These are values as disclosed in the selected 10-K. Purchase-price allocations can be preliminary; subsequent quarterly revisions are not retrieved by this section.


In [ ]:
if acquisition_tables:
    display(acquisition_tables["nwc"].round(2))
else:
    print("Load supported acquisition disclosures in section 8 before using this bridge.")


### 8b. Review transaction expenses without changing the historical statements

The candidate column removes only acquisition costs explicitly tagged to **G&A**. Interest transaction costs are shown separately in the deal table and are not added to EBIT or EBITDA. The broader discussion of professional services and other expense changes is not treated as an acquisition-cost addback.

This is an illustrative expense adjustment, not an approved recurring cost assumption or a full-year combined-company forecast. Annual production still includes acquired output only after closing. Historical `annual`, `operating` and `drivers` values remain the reported series.


In [ ]:
if acquisition_tables:
    display(acquisition_tables["expense_candidates"].round(2))
    print("Candidate only: decide whether the disclosed expense is appropriate to exclude before forecasting.")
else:
    print("Expense candidates are unavailable until the acquisition disclosures are loaded.")


### 8c. Distinguish post-closing results from full-year pro forma information

The company's **unaudited pro forma** revenue and net income describe a hypothetical earlier closing. They are not observed full-year acquired production, EBITDA or free cash flow. Do not annualize the post-closing contribution by multiplying by a fraction of the year.

In the audit table, an acquiree's since-closing contribution can have a full-year XBRL context. Its economic coverage still begins on the acquisition date shown in the deal table. The metric name and source note govern that distinction.


In [ ]:
if acquisition_tables:
    display(acquisition_tables["proforma"].round(2))
    display(Markdown("**Extraction audit: source tags, transaction dimensions, dates and unit scales**"))
    display(acquisition_audit)
else:
    print("No supported acquisition contribution or pro forma disclosures are available.")


### What this means for the next step

We have a sourced acquisition bridge and a visible expense adjustment candidate. We still need a forecast base that reflects the assets owned for the forecast period.

1. Build the oil, gas and NGL volume/realized-price revenue bridge. Separate upstream sales from marketing activity and derivatives.
2. Refresh production disclosures and current guidance, including acquired operations and subsequent asset sales. Do not label reported production growth as organic without supporting data.
3. Review capital payables and cash-flow working-capital disclosures, plus recurring unit costs, DD&A and capital needs. Capex per Boe alone does not establish sustaining capex or development economics.
4. Then select explicit production, commodity price, cost, tax, capex and NWC assumptions to calculate forecast UFCF. WACC and valuation follow those operating forecasts.

Gas-to-Boe conversion compares energy content, not dollar value. The forecast will model the three commodities separately.


## 9. Oil, natural gas and NGL revenue model: historical reconstruction

We now connect operating quantities to revenue, using the same economic relationships needed for the forecast. **These are historical calculations; no future growth or commodity prices have been assumed.**

This section retrieves daily production, composite realized prices, reported commodity sales and other revenue components directly from annual filing documents. It uses each report's comparative years to cover the selected history window. The observations are new, independently sourced inputs called `revenue_disclosures`; they do not overwrite the earlier operating reference dataset.

All financial amounts below are **USD millions**, unless another unit is shown. Crude oil includes condensate. Prices exclude the financial derivative effects reported separately in the income statement.

`REFRESH_REVENUE_FILINGS = False` uses saved documents. To retrieve documents for a newly selected window, refresh SEC metadata in the setup cell first, set this flag to `True` once, then return it to `False`. No financial values or fixed year lists are embedded in the code.


In [ ]:
import app.eog_revenue as revenue_source_module
import app.energy_revenue as revenue_model_module
importlib.reload(revenue_source_module)
importlib.reload(revenue_model_module)

REFRESH_REVENUE_FILINGS = False
revenue_disclosures = None
revenue_precision = None
revenue_audit = pd.DataFrame()
commodity_revenue = None
revenue_bridge = None

try:
    if history.provider.upper() != "SEC":
        raise ValueError("Use FINANCIAL_SOURCE = 'sec' for this filing revenue bridge.")
    revenue_disclosures, revenue_precision, revenue_audit = revenue_source_module.load_revenue_history(
        annual.index, cik=SEC_CIK, ticker=TICKER,
        refresh=REFRESH_REVENUE_FILINGS, root=PROJECT_ROOT,
    )
    commodity_revenue, revenue_bridge = revenue_model_module.build_revenue_bridge(
        revenue_disclosures, revenue_precision, annual, provider=history.provider,
    )
    display(Markdown("**Production and realized prices extracted from the filings**"))
    display(revenue_disclosures[revenue_source_module.OPERATING_FIELDS].T.round(2))
    display(Markdown("**Source coverage by fiscal year**"))
    display(revenue_audit[["year", "filing_report_date", "accession", "filed", "retrieved_at_utc"]].drop_duplicates().sort_values("year"))
except (SECError, ValueError, OSError) as error:
    print(f"Revenue model unavailable: {error}")


### 9a. Calculate sales from production and realized prices

| Commodity | Annual volume calculation | Revenue in USD millions |
|---|---|---|
| Oil and condensate | MBbl/day x days / 1,000 = MMBbl | MMBbl x USD/Bbl |
| Natural gas | MMcf/day x days / 1,000 = Bcf | Bcf x USD/Mcf |
| NGLs | MBbl/day x days / 1,000 = MMBbl | MMBbl x USD/Bbl |

The gas formula works because one Bcf contains one million Mcf. **No Boe conversion is used to price gas.** Days are calculated from each fiscal year, including leap years. Acquired production is already included in the reported daily average only for the period owned; do not add it a second time or annualize it here.

Published daily volumes and prices are rounded. `Calculated less reported` retains the resulting difference. The displayed rounding bound assumes rounding to the nearest published unit for volume, price and reported sales:

`days / 1,000 x (abs(volume) x half-price-unit + abs(price) x half-volume-unit + half-volume-unit x half-price-unit) + half-sales-unit`

A difference within this bound is consistent with published rounding; it does not prove rounding is the only cause. Missing inputs stay unavailable, and larger differences remain flagged for review.


In [ ]:
if commodity_revenue is not None:
    display(commodity_revenue.round(3))
    print(commodity_revenue["Status"].value_counts().to_string())
    print("Calculated sales, reported sales, differences and rounding bounds are USD millions.")
else:
    print("Load the section 9 revenue disclosures first.")


### 9b. Reconcile upstream sales to total company revenue

The first reconciliation uses **reported oil + gas + NGL sales**. Add gathering/processing/marketing revenue, signed derivative gains or losses, signed asset-disposition gains or losses, and other revenue to obtain the filed total. The two residual rows show whether that total matches both the component sum and our separate SEC financial-history table.

Marketing revenue includes third-party commodity sales, fees and sand activity. Marketing costs are shown separately below the revenue bridge. **Marketing revenue less marketing costs is only a subtotal**, not standalone segment EBITDA; other costs can be classified elsewhere. If we later model marketing on a net basis, we must also remove the corresponding marketing costs from the expense build to avoid double counting.

Financial derivative gains/losses include cash settlements and valuation changes. The signed income-statement amount is used only to reconcile historical revenue; it is not automatically a recurring revenue or cash-flow forecast. Asset-sale gains also differ from cash sale proceeds.


In [ ]:
if revenue_bridge is not None:
    display(revenue_bridge.T.round(3))
    revenue_reconciliation = revenue_bridge[[
        "Component sum less filing total", "Filing total less SEC annual history",
    ]].copy()
    revenue_reconciliation["Status"] = revenue_reconciliation.apply(
        lambda row: "Missing input" if row.isna().any() else
        "Ties" if row.abs().le(0.05).all() else "Review difference", axis=1,
    )
    display(revenue_reconciliation)
    print("Each residual is USD millions; reconciliation tolerance is $0.05 million.")
else:
    print("Revenue reconciliation is unavailable until section 9 loads successfully.")


### 9c. Trace the inputs and prepare the forecast decisions

The audit records filing, reporting period, original text, scale, units, published precision, and the XBRL product dimension or labeled table row. All fields for a year come from one selected filing. Earlier comparative years may have later restatements; the separate SEC annual-total comparison makes total-revenue differences visible.

The supported filing layouts now automate retrieval of these six production/price inputs and the revenue components. Reserves, unit costs and the other earlier operating observations still require their own refresh workflow. An unfamiliar layout or missing tag requires review rather than a substituted value.

**Next:** establish forward oil, gas and NGL volumes using current production/guidance and the assets owned for the forecast period, then choose realized-price assumptions and differentials. Encino acquisition timing and subsequent asset sales still matter. The historical growth rates and rounded-price reconciliation differences are not forecast assumptions.


In [ ]:
if not revenue_audit.empty:
    display(revenue_audit.sort_values(["year", "metric"]))
else:
    print("No revenue source audit is available.")


## Refresh next year

The year range advances when a new annual period appears in a refreshed source. It does not advance merely because the calendar changes. To display five years and calculate the earliest change, retrieve six annual records.

**WRDS:** In a separate PowerShell window at the project root, run:

```powershell
.\.venv-wrds\Scripts\python.exe -m app.refresh_wrds --ticker EOG --years 6
```

Sign in through the WRDS prompts and approve Duo. Then rerun this notebook with `FINANCIAL_SOURCE = "wrds"`.

**SEC:** Add `SEC_USER_AGENT=DCF-Model your-contact-email` to your existing local `.env`, replacing the contact text with your email. Do not overwrite the existing file or paste credentials into the notebook. Set `FINANCIAL_SOURCE = "sec"` and `REFRESH_SEC = True`, then run the first cell. After a successful download set `REFRESH_SEC = False` for cached exploration. No SEC API key is needed. The SEC request includes your configured contact header.

SEC's [API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces) explains the standard concepts it exposes; custom company disclosures need a separate adapter. This notebook's SEC mapping is conservative, so unsupported tags remain blank and visible in the coverage table.

The original root-level `EOG_Financials.ipynb` remains your prior static research notebook. Continue new dynamic work here to avoid reintroducing its hardcoded arrays.